# Notebook 6B— More trees are often better than one: Random Forest models

*Authored by Dr. Noelle Anderson in 2026*

## Introduction

*This is your final notebook. Take a moment to appreciate how far you have come, and get ready for your final day!*

In Notebook 6A, you worked with a single decision tree. A single tree is useful because you can inspect its decision rules, but you also saw a tradeoff: a very small tree may miss useful patterns, while a very deep tree can become difficult to interpret and may fit the training data too closely.

Today, you will learn a related model called a **random forest**. A random forest combines many decision trees instead of relying on just one. This makes it useful for many table-shaped biology, health, and environmental datasets.

You will use the same `High Asthma` classification target from Week 5 and Notebook 6A. First, you will fit a random forest using a narrower environmental-exposure feature set, similar to the first supervised models you built in Week 4. Then you will compare that model with an expanded-context model that also includes socioeconomic, age-structure, and racial/ethnic demographic context variables.

This comparison is about how feature choice shapes a machine-learning problem. An environmental-only model asks a more specific question about exposure variables. An expanded-context model asks what changes when the model also has access to tract-level social context. In public-health and environmental-justice work, both questions can be useful, but they need different interpretations.

Because this dataset describes real places where people live, learn, work, breathe, and care for each other, we will interpret the results with care. The social and demographic variables describe census-tract patterns, not individual people.

At the end, you will compare **feature importance** for the two random forest models. You saw coefficients in earlier linear models. Feature importance is another way to ask what a fitted model relied on, but it does not work the same way as coefficients and needs careful interpretation.

By the end of this notebook, you will be able to explain random forests, fit and evaluate `RF_model`, compare feature sets, compare feature importance results, and interpret the results carefully in an environmental-justice context.

## Notebook 6B roadmap

Today you will work through five parts:

1. **Prepare the data** by recreating the `High Asthma` target, choosing model features, splitting the data, and imputing missing values.
2. **Understand ensembles, bagging, and random forests** using the decision-tree ideas from Notebook 6A.
3. **Fit and evaluate a random forest** called `RF_model` using environmental-exposure features.
4. **Compare two feature sets** by fitting a second random forest called `RF_model_expanded` with a fuller contextual feature set.
5. **Compare feature importance and interpret limitations** in context.

# Part 1: Prepare the data

We will start with the same workflow you have used in the previous supervised-learning notebooks:

1. Load the data.
2. Create the target.
3. Choose the model features.
4. Split the data into training and test sets.
5. Impute missing feature values.

Most of this is review, so the explanations will be shorter than they were earlier in the course.

## Step 0: Import the packages we need

The new import today is `RandomForestClassifier`, which lets us build a random forest classification model.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# our preprocessing
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

# our model
from sklearn.ensemble import RandomForestClassifier # new today!

# our metrics
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.metrics import ConfusionMatrixDisplay

## Step 1: Load the CalEnviroScreen data

We will load the same public CalEnviroScreen dataset from Google Drive. Use `Census Tract` as the index column, just like you did in the previous CalEnviroScreen notebooks.

The loading code is provided so we can focus today on random forests and feature choice.

In [ ]:
# Instructor-provided data loading code
file_id = "1X4-6X3VKhR2jRHppI3XuVV79nhHyg8Xb"
url = f"https://drive.google.com/uc?export=download&id={file_id}"

ces = pd.read_csv(url, index_col="Census Tract")

display(ces.head())

## Step 2: Prepare rows with known `Asthma` values

For supervised learning, every row used for training needs a known target value. Today, our target will be created from the `Asthma` column, so we need to remove rows where `Asthma` is missing.

In [ ]:
print("Original shape:", ces.shape)

# Drop rows where the Asthma value is missing
ces_sup_clean = ces.dropna(subset=["Asthma"]).copy()

print("Shape after dropping rows with missing Asthma:", ces_sup_clean.shape)

## Step 3: Create the `High Asthma` classification target

We will use the same binary target from Week 5 and Notebook 6A:

- tracts at or above the median `Asthma` value are labeled class `1`
- tracts below the median `Asthma` value are labeled class `0`

Class `1` means actual or predicted **high asthma burden**. Class `0` means actual or predicted **lower asthma burden**.

In [ ]:
# Find the median Asthma value
asthma_cutoff = ces_sup_clean["Asthma"].median()

# Create the binary classification target
ces_sup_clean["High Asthma"] = (ces_sup_clean["Asthma"] >= asthma_cutoff).astype(int)

print("Asthma cutoff:", asthma_cutoff)

print("\nClass counts:")
print(ces_sup_clean["High Asthma"].value_counts())

print("\nClass proportions:")
print(ces_sup_clean["High Asthma"].value_counts(normalize=True))

## Step 4: Choose the environmental feature set

We will start with an environmental-exposure feature set, similar to the narrower feature set used in your first supervised models in Week 4.

This gives us a clean first modeling question:

> Can a random forest classify census tracts as high asthma burden using environmental exposure variables?

This first model is intentionally limited. It does not include socioeconomic, age-structure, or racial and ethnic demographic context variables. That does not mean those context variables are unimportant. We are starting with the narrower feature set so that we can later compare it with a fuller contextual model and see how the feature choice changes performance and interpretation.

In [ ]:
# Instructor-provided environmental-exposure feature set
model_features = [
    "Ozone",
    "PM2.5",
    "Diesel PM",
    "Drinking Water",
    "Pesticides",
    "Traffic",
    "Cleanup Sites"
]

print("Number of main model features:", len(model_features))
print(model_features)

## Step 5: Create `X` and `y`

Now create the feature matrix `X` and the target labels `y`.

Reminder:

- `X` contains the features the model will use to make predictions.
- `y` contains the target the model is trying to predict.

### <font color=0D9488>**Question 1:**</font> Create the `X` feature dataset using `model_features` and create the `y` target labels using `High Asthma`. Then print the shapes of both objects.

In [ ]:
# Your solution here!

## Step 6: Split the rows into training and test sets

Use `train_test_split()` again. Because this is classification, we will use `stratify=y`, just like in Notebook 5B and Notebook 6A.

`stratify=y` asks sklearn to keep the class proportions similar in the training set and the test set. This dataset is already close to balanced because we used the median cutoff, but stratifying is still good practice for classification.

### <font color=0D9488>**Question 2:**</font> Split `X` and `y` into training and test sets using 20% test data, `random_state=42`, and `stratify=y`. Then print the shapes of the four resulting objects and check the class proportions in `y_train` and `y_test`.

In [ ]:
# Your solution here!

## Step 7: Impute missing feature values

Random forests do not need feature scaling, but sklearn's basic random forest still needs complete numerical input data. That means we still need to handle missing feature values before fitting the model.

Use the same median imputation workflow from the previous notebooks:

1. Fit the imputer on the training features only.
2. Transform both the training features and the test features.
3. Convert the imputed arrays back into dataframes with readable column names.

Fitting the imputer only on the training data helps keep information from the test set out of the training process.

### <font color=0D9488>**Question 3:**</font> Use `SimpleImputer` to impute missing values in `X_train` and `X_test` with the median from `X_train`. Convert the arrays to final dataframes named `X_train_imputed` and `X_test_imputed`, then check that each dataframe has 0 missing values.

In [ ]:
# Your solution here!

# Part 2: Understand ensembles, bagging, and random forests

Notebook 6A ended with a problem: one decision tree can be readable, but it can also be too sensitive to the exact rows and splits it learned from.

A **random forest** responds to that problem by fitting *many* decision trees, a whole forest of trees, and *combines* their predictions. This kind of model that relies on many smaller models is called an **ensemble**.

An **ensemble model** combines multiple models to make one final prediction. The basic idea is that many imperfect models can work together to make a more stable prediction than one model alone, just like how diverse groups of people are better at decision making than one person alone! The power of many!


## Step 8: What is bootstrapping?

Before we define a random forest, we need one general statistics idea: **bootstrapping**.

Imagine physically cutting out each training row as a strip of paper and putting all the strips into a bowl. To make a bootstrap sample, you close your eyes, pull out one paper strip, make a copy of it to put into a new pile, and then put the original strip back into the bowl before picking again.

Because the paper strip goes back into the bowl each time, the same row can be picked more than once. Some rows may not be picked at all. This is called sampling **with replacement**.

In a random forest, each tree gets a different bootstrap sample of the training rows. This gives different trees slightly different versions of the training data, which helps the trees avoid all making exactly the same decisions.

## Step 9: What is bagging?

Bootstrapping helps us make many trees that are slightly different from each other. Each tree sees a different bootstrap sample of the training rows, so each tree may learn slightly different splits.

But that creates a new question: how do we get just **one** answer from a whole forest of trees, each with its own prediction?

That is where **bagging** comes in. Bagging stands for **bootstrap aggregating**.

- **Bootstrap**: train each model on a different bootstrap sample of the training rows.
- **Aggregate**: combine the predictions from many models into one final prediction.

The aggregation step depends on the type of prediction task:

- For **classification**, the trees can be thought of as voting. If most trees vote for `High Asthma`, the forest predicts `High Asthma`. If most trees vote for `Lower Asthma`, the forest predicts `Lower Asthma`.
- For **regression**, where the model predicts a number, the predictions are usually averaged. For example, if several trees each predict a different asthma rate, the forest can average those predictions into one final prediction.

Bagging helps because one tree may fit details that depend on the exact training rows it saw. When many trees see slightly different training samples, their individual quirks can partly cancel out when their predictions are combined.

## Step 10: What makes a random forest random?

The randomness in a random forest is not just noise. It is what helps the forest avoid making 100 copies of the same tree.

A single decision tree can be sensitive to the exact rows and features it uses. If many trees all saw the same rows and considered the same features in the same order, they would probably learn very similar splits. Combining many nearly identical trees would not help much.

A random forest uses two main sources of randomness to make the trees different from each other.

**Source 1: random samples of rows**

Each tree trains on a bootstrap sample of the training data. This means each tree sees a slightly different set of rows.

**Source 2: random subsets of features**

When a tree is choosing a split, it does not always consider every feature, it considers a random subset of features. This helps prevent every tree from relying on the same strongest feature in the same way.

These two sources of randomness make the trees different from each other and this difference is so valuable to us! The forest combines many different trees that used different rows and features instead of just one tree or repeating the same tree 100 times. When the trees make somewhat different mistakes, combining them can make the final prediction more stable than relying on one tree only.

**A forest is often stronger than one tree alone! We are all stronger together!**

## Step 11: Why use a random forest model?

Random forests are popular because they often work well on table-shaped (tabular) datasets like this one. They can learn more complex nonlinear patterns and use different combinations of features.

Random forests are widely used in biology, biomedicine, conservation, and public health because many real datasets have complicated patterns. In public-health work, a model can support decision-making, but it should not replace community expertise, clinical judgment, or policy analysis.

They also connect to the underfitting and overfitting ideas from Notebook 6A. A single shallow tree may underfit because it is too simple. A single deep tree may fit training-specific details. A random forest trains many trees and combines them, which often makes the final prediction more stable than relying on one tree.

Random forests can still overfit, and they are not automatically the best model. Their main tradeoff is interpretability. One small tree is a flowchart we can inspect, like in our last notebook. A forest of many trees is harder to explain directly, so today we will use metrics and feature importance to help us understand model performance and model behavior.



## Step 12: How many trees are in the forest?

A random forest has several settings that we choose before fitting the model. In machine learning, these model settings are called **hyperparameters**, just like `max_depth` was a hyperparameter for single decision trees.

One of the main random forest hyperparameters is `n_estimators`.

`n_estimators` controls the number of trees in the forest. A larger number of trees can make predictions more stable, but it also takes more time to fit the model.

Today we will use `n_estimators=100`, which means the random forest will fit 100 decision trees. In a longer project, you might tune this value and other hyperparameters. Tuning is outside the scope of this accelerated course, so we will use one practical starting value.

As a reminder from Notebook 6A, we should not keep changing model settings after checking performance on the test set. The test set is supposed to act like new, unseen data. In a full project, model settings and feature-set choices are usually compared with a validation set or cross-validation first, and the test set is saved for the final evaluation.

# Part 3: Fit and evaluate a random forest

Now we will fit our first random forest model!

We will call it `RF_model` and this first model will use the environmental-exposure feature set in `model_features` from Step 4.

## Step 13: Fit `RF_model`

The code pattern should look familiar:

1. Create the model object.
2. Fit the model on the training data.

The new model class is `RandomForestClassifier`. Because this is your first random forest, the code is provided. After this example, you will fit another random forest yourself later on.


In [ ]:
# Instructor-provided random forest model
# n_estimators is the number of trees in the forest
# random_state makes the random sampling reproducible
RF_model = RandomForestClassifier(n_estimators=100, random_state=42)

# Fit/Train the model on the training data
RF_model.fit(X_train_imputed, y_train)

## Step 14: Make predictions and calculate metrics

Now use the fitted random forest model to predict classes for the test data.

After making predictions, you will calculate the same classification metrics from Week 5 and Notebook 6A:

- accuracy
- precision
- recall
- F1

Below is a multi-step question, take it one part at a time!

### <font color=0D9488>**Question 4:**</font> Use `RF_model` to make predictions for `X_test_imputed`. Save the predictions as `RF_preds`.

In [ ]:
# Your solution here!

### <font color=0D9488>**Question 5:**</font> Using the predictions above, calculate accuracy, precision, recall, and F1. Store the results in a dataframe called `RF_metrics` and display it.

In [ ]:
# Your solution here!

You should see pretty good metrics here in the 80%s!

## Step 15: Plot the confusion matrix

The confusion matrix shows which test rows were classified correctly and which were classified incorrectly.

This is especially important in public-health settings because different error types can have different consequences. For this asthma-burden example, a false negative would mean a tract was actually high asthma burden, but the model predicted lower asthma burden.

We can think about what these could lead to:  A false positive could mean that a tract is flagged for follow-up even though it is not in the high-asthma group in this dataset. Below you'll be asked to think about what a false negative means.

Model errors are not just numbers., these errors could affect where resources are directed, and which places receive follow-up or are missed. That is why we look beyond overall accuracy!

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    RF_preds,
    display_labels=["Lower Asthma", "High Asthma"], # Can add nice labels here
    cmap="Blues"
)

plt.title("Confusion Matrix for the Random Forest Model")
plt.show()

### <font color=0D9488>**Question 6:**</font> Look at the confusion matrix. In this public-health context, what would a false negative mean? Why might false negatives be especially important if the model were used to help identify places that may need additional asthma-related resources?

**Your solution here!**

# Part 4: Compare two feature sets

So far, `RF_model` used only environmental-exposure features. That gave us a clean first model, but it left out variables that may matter for understanding asthma burden across places.

Now we will fit an expanded random forest with environmental-exposure variables plus socioeconomic, age-structure, and racial and ethnic demographic context variables.

This comparison asks a different modeling question:

> What changes when the random forest has access to both environmental exposures and broader tract-level context?

The racial and ethnic demographic columns require especially careful interpretation. These columns describe population patterns across census tracts and those patterns are shaped by history, policy, housing access, environmental racism, economic inequality, migration, and other social conditions. They cannot accurately or appropriately be interpreted as biological explanations for asthma burden or as descriptions of individual people.

A model with more features is not automatically better or more appropriate! We need to compare performance, examine what the model relied on, and interpret the feature choice carefully.


## Step 16: Define the expanded feature set

The environmental-exposure feature set is still called `model_features`.

Now we will add a new list called `additional_context_features`. This list includes socioeconomic context variables, age-structure variables, and racial and ethnic demographic context variables.

Then we will combine the two lists into `expanded_model_features`.

In [ ]:
# Instructor-provided additional context features
additional_context_features = [
    "Education",
    "Housing Burden",
    "Linguistic Isolation",
    "Poverty",
    "Unemployment",
    "Children < 10 years (%)",
    "Elderly > 64 years (%)",
    "Hispanic (%)",
    "White (%)",
    "Black (%)",
    "Native American (%)",
    "Asian American (%)",
    "Other/Multiple (%)"
]

# Expanded feature set for the comparison model
expanded_model_features = model_features + additional_context_features

print("Number of main model features:", len(model_features))
print("Number of additional context features:", len(additional_context_features))
print("Total number of expanded model features:", len(expanded_model_features))
print(expanded_model_features)

## Step 17: Prepare the expanded training and test data

We want this to be a fair comparison, so we will use the same training rows and the same test rows as before. We do this because the question we want to ask is about the **features**, not about a different random split of the data.

If we changed the training and test rows at the same time that we changed the feature set, we would be changing two things at once:

1. which census tracts the model trains on and tests on
2. which columns the model gets to use

That would make the comparison harder to interpret. If the model performance changed, we would not know whether it changed because of the expanded feature set, because of the different train/test split, or both.

To avoid that problem, we will keep the same rows and only change the columns. We will use the existing `X_train.index` and `X_test.index` to select the same census tracts from the expanded feature dataset.

Then we will impute the expanded features using a new imputer. We need a new imputer because the expanded feature set has additional columns, and each column needs its own training-set median.

The code is provided because the main goal here is not another preprocessing challenge. The main goal is to compare what happens when the model has access to a different set of features while the training rows, test rows, target, model type, and evaluation metrics stay the same.

In [ ]:
# Instructor-provided expanded feature preparation

# Create the expanded feature dataset
X_expanded = ces_sup_clean[expanded_model_features]

# Use the same training and test rows as the main model
X_train_expanded = X_expanded.loc[X_train.index]
X_test_expanded = X_expanded.loc[X_test.index]

# Create a new imputer for the expanded feature set
expanded_imputer = SimpleImputer(strategy="median")

# Fit the imputer on the expanded training features only
X_train_expanded_imputed_array = expanded_imputer.fit_transform(X_train_expanded)

# Apply the trained imputer to the expanded test features
X_test_expanded_imputed_array = expanded_imputer.transform(X_test_expanded)

# Convert the arrays back to dataframes
X_train_expanded_imputed = pd.DataFrame(
    X_train_expanded_imputed_array,
    columns=expanded_model_features,
    index=X_train_expanded.index
)

X_test_expanded_imputed = pd.DataFrame(
    X_test_expanded_imputed_array,
    columns=expanded_model_features,
    index=X_test_expanded.index
)

print("X_train_expanded_imputed shape:", X_train_expanded_imputed.shape)
print("X_test_expanded_imputed shape:", X_test_expanded_imputed.shape)

print("\nMissing values in X_train_expanded_imputed:")
print(X_train_expanded_imputed.isna().sum())

print("\nMissing values in X_test_expanded_imputed:")
print(X_test_expanded_imputed.isna().sum())


## Step 18: Fit and evaluate the expanded random forest

Now fit a second random forest called `RF_model_expanded`.

To keep the comparison fair, use the same random forest settings:

- `n_estimators=100`
- `random_state=42`

The difference is the feature set. `RF_model` used the environmental-exposure feature set. `RF_model_expanded` will use the expanded contextual feature set.

### <font color=0D9488>**Question 7:**</font> Create and fit `RF_model_expanded` using the new expanded and imputed training data, and the same number of trees as before.

In [ ]:
# Your solution here!

### <font color=0D9488>**Question 8:**</font> Now on that trained model, make test predictions, calculate accuracy, precision, recall, and F1, and create and print comparison table with the metrics for `RF_model` and `RF_model_expanded`.

In [ ]:
# Your solution here!

### <font color=0D9488>**Question 9:**</font> The two models used different feature sets. Which model had better test-set performance in this run? Why might someone still want to study the expanded-context model, even though it did not perform better here?

**Your solution here!**

# Part 5: Compare feature importance and interpret limitations

So far, we compared the two random forest models using classification metrics. Metrics tell us how the models performed on the test set, but they do not show which features the models used most while learning.

Random forests have a built-in **feature importance** score. Feature importance tells us how much each feature helped the forest make useful splits across many trees. Here, “useful” means the split helped separate rows with different target labels, like `Lower Asthma` and `High Asthma`.


This may remind you of the coefficients you saw in linear and logistic regression because both help us ask, “Which features did the model rely on?” They answer that question in different ways.

A coefficient describes how a feature is associated with the model’s prediction in a linear model, while holding the other model features constant. Coefficients can be positive or negative, but they still do not prove causation.

Random forest feature importance gives a ranking of how much the model used each feature when making useful splits across many trees. It does not tell us whether larger values of a feature push predictions toward `High Asthma` or `Lower Asthma`.

Feature importance also depends on which features the model was allowed to use. The environmental-only model can only assign importance to environmental-exposure variables. The expanded model can assign importance to a wider set of context variables.

When features describe social conditions or community demographics, feature importance needs careful interpretation. A high importance score describes what this fitted model relied on. It is one tool for understanding model behavior, not a complete public-health explanation.

## Step 19: Get feature importance for the environmental-only random forest

First, we will look at feature importance for `RF_model` which used only the environmental-exposure features in `model_features`, by doing `RF_model.feature_importances_`

The code is provided because this is your first time seeing feature importance. Focus on how the table is made, then you will adapt the same pattern for your second model.


In [ ]:
# Instructor-provided feature importance for the environmental-only random forest
RF_importance = pd.DataFrame({
    "Feature": model_features,
    "Importance": RF_model.feature_importances_ # notice how we extract feature importance!
})

RF_importance = RF_importance.sort_values(
    by="Importance",
    ascending=False
)

display(RF_importance)

In [ ]:
# Instructor-provided plot for the environmental-only random forest
plt.figure(figsize=(8, 5))

sns.barplot(
    data=RF_importance,
    x="Importance",
    y="Feature"
)

plt.title("Feature Importances in RF_model")
plt.xlabel("Feature importance")
plt.ylabel("Feature")
plt.show()


Here we can see that the model relied most on `PM2.5`, `Ozone`, and `Drinking Water` among the environmental-exposure features.

This does not mean these features cause high asthma burden, and it does not tell us whether higher values increase or decrease the prediction. It means that, within this environmental-only random forest, these features were used most often to make useful splits across the trees.

## Step 20: Get feature importance for the expanded random forest

Now you will repeat the same process for `RF_model_expanded`, which used environmental-exposure features plus additional tract-level context features.

Use the same pattern from Step 19, but replace:

- `model_features` with `expanded_model_features`
- `RF_model.feature_importances_` with `RF_model_expanded.feature_importances_`
- `RF_importance` with `expanded_RF_importance`

Then plot the top 10 features from the expanded model. We use the top 10 because the expanded model has 20 features, and plotting all of them would be harder to read.


### <font color=0D9488>**Question 10:**</font> Create a feature-importance dataframe for `RF_model_expanded`, sort it from highest to lowest importance, display it, and plot the top 10 features using a horizontal bar plot.


In [ ]:
# Your solution here!

### <font color=0D9488>**Question 11:**</font> Compare the feature importance plots for `RF_model` and `RF_model_expanded`. How did the most important features change when the model was allowed to use socioeconomic, age-structure, and racial/ethnic demographic context variables? What does this tell us about how feature choice can change what a model relies on?

**Your solution here!**

## Final reflection

In this notebook, the environmental-only random forest and the expanded random forest answered different modeling questions. The environmental-only model asked whether exposure variables alone could classify census tracts as high asthma burden. The expanded model asked what changed when the model also had access to socioeconomic, age-structure, and racial/ethnic demographic context variables.

In these results, the environmental-only model performed slightly better on the test set, even though the expanded model had more features. At the same time, the feature-importance plots showed that the expanded model relied heavily on several social context variables, including variables like `Education`, `Hispanic (%)`, `Poverty`, `Black (%)`, and `White (%)`. Together, these results show an important machine-learning lesson: more features can change what a model uses, but they do not automatically improve prediction on new data.

These results also deserve careful, respectful interpretation. This dataset describes real places where we live, learn, work, breathe, and care for each other. The racial, ethnic, and socioeconomic variables describe tract-level patterns shaped by history, policy, housing access, environmental racism, economic inequality, migration, and other social conditions. They are not biological explanations for asthma burden, and they are not descriptions of individual people.

The model is learning associations across census tracts. It can help us ask where patterns appear in the data, but it cannot explain anyone’s personal health, family, neighborhood, or lived experience. It also cannot decide what a community needs. Those questions require local knowledge, public-health expertise, historical context, and the voices of people most affected by the conditions being studied.

Before using a model like this outside a summer course, we would need to ask more questions: What decision would this model support? Who could benefit if the model works well? Who could be harmed if it makes mistakes? Are errors distributed unevenly across places? What information is missing from the dataset? What community knowledge or public-health expertise should guide interpretation?

Good machine learning requires more than a higher metric or a feature-importance ranking. It also requires understanding what the model is using, what it cannot tell us, and what responsibilities come with using predictions in the real world.

# Congratulations, you have completed today's notebook and the SCIP course!

YAYYYY— Give yourself and your team a round of applause for how far you've come!

## Key Takeaways from Today:

- You fit a random forest classifier called `RF_model`.
- You learned that a random forest is an ensemble made of many decision trees.
- You learned that random forests use bootstrapped samples of rows and random subsets of features.
- You connected random forests to the underfitting, overfitting, and interpretability ideas from Notebook 6A.
- You evaluated a random forest using familiar classification metrics.
- You compared an environmental-only feature set with an expanded contextual feature set.
- You compared feature importance for the environmental-only and expanded random forest models.
- You practiced interpreting feature choice carefully in a public-health and environmental-justice context.

Random forests are useful because they combine many decision trees into a more stable model. They often perform well on table-shaped datasets, but they are harder to interpret than one small tree. In this notebook, you also saw that feature choice is part of the modeling question. An environmental-only model and a fuller contextual model answer different questions, and those choices affect both performance and interpretation.

## Where Today Fits in the ML Workflow

Across SCIP, you have now seen unsupervised learning, regression, classification, single decision trees, and ensemble models. The same core habits carry across these methods: define the question clearly, prepare the data carefully, evaluate on held-out data, and interpret results in context.

Notebook 6A introduced single decision trees as interpretable nonlinear models. Notebook 6B introduced random forests as ensembles of many trees and used them to examine feature choice. Together, these notebooks show a central idea in machine learning: model choice and feature choice both affect performance, interpretability, and real-world meaning.


# Wrapping up the whole course

Across SCIP, you have built a real foundation in machine learning, and you should be proud of how much you can now do! You have used unsupervised learning to look for groups in data, trained supervised models to predict numerical and categorical targets, prepared data with train/test splits and preprocessing, evaluated models with metrics, visualized results, and interpreted models carefully.

You have also practiced one of the most important ML habits: starting with a clear question. As you keep going, try turning your own scientific or public-health interests into ML questions!

You might ask unsupervised questions like, “Are there groups of cells with similar gene-expression patterns?”, “Do samples cluster by treatment condition?”, or “Are there neighborhoods with similar environmental-burden profiles?” You might ask supervised questions like, “Can we predict whether a cell is responding to a drug?”, “Can we predict enzyme activity from molecular features?”, “Can we predict disease risk from clinical and exposure variables?”, or “Can we predict water quality from environmental measurements?”

Whatever question you ask, keep returning to the same core habits: define the target if there is one, choose features thoughtfully, split and preprocess data carefully, evaluate the model on held-out data, and interpret the results in context. Think about what errors would matter most, who could be affected by those errors, and what the model can and cannot tell you.

Thanks for sticking with us this summer. Celebrate your accomplishment and go forth and model responsibly!